# Household energy consumption and clustering

This notebook sums energy from **all complete minute readings** and compares three clustering methods on the **same reproducible sample**. Read the repository README for data provenance, units, and limitations.


In [ ]:
import os
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    raise RuntimeError("Start Jupyter from the repository root (the directory containing src/).")
sys.path.insert(0, str(repo_root))

from src.energy_analysis import CLUSTER_FEATURES, cluster_sample, compare_clusters, load_power_data, monthly_energy

data_path = Path(os.environ.get("ENERGY_DATA_PATH", repo_root / "data" / "household_power_consumption.txt"))
if not data_path.is_file():
    raise FileNotFoundError(f"Dataset missing: {data_path}. See README.md for download instructions.")


## 1. Load and clean

Missing measurements are removed before sampling. Filling after a random sample would combine unrelated minutes.


In [ ]:
readings, row_counts = load_power_data(data_path)
print(row_counts)
readings.head()


## 2. Observed monthly energy

The total and sub-metering values below are in kWh. These are sums of observed complete minutes, so use coverage when comparing partial months or months with missing values. `Other_kWh` is a calculated residual.


In [ ]:
monthly = monthly_energy(readings)
display(monthly[["Total_kWh", "observed_minutes", "coverage"]].round(3))
monthly_for_plot = monthly[["Sub_metering_1_kWh", "Sub_metering_2_kWh", "Sub_metering_3_kWh", "Other_kWh"]].copy()
monthly_for_plot.index = monthly_for_plot.index.strftime("%Y-%m")
monthly_for_plot.plot(
    kind="bar", stacked=True, figsize=(14, 5), title="Observed monthly household energy"
)
plt.ylabel("kWh from complete minute readings")
plt.xlabel("Month")
plt.tight_layout()
plt.show()


## 3. Prepare a reproducible clustering sample

Models use the raw measured features listed below. Total energy and the derived residual are excluded to avoid feeding the same quantity in multiple forms. The sample is for clustering only; monthly energy above uses all complete readings.


In [ ]:
sample, scaled = cluster_sample(readings, sample_size=5000, random_state=42)
print(f"Clustering sample: {len(sample):,} rows")
print("Features:", CLUSTER_FEATURES)
display(sample[CLUSTER_FEATURES].describe().round(2))


## 4. Compare clustering methods

The elbow chart is a guide for choosing K. The comparison below starts at five clusters and uses an exploratory DBSCAN radius of 1.5. Both should be revisited after examining the score table and cluster profiles.


In [ ]:
ks = range(2, 11)
inertia = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(scaled).inertia_ for k in ks]
plt.plot(list(ks), inertia, marker="o")
plt.xlabel("K-Means clusters")
plt.ylabel("Inertia")
plt.title("K-Means elbow guide")
plt.show()

scores, labels_by_model = compare_clusters(scaled, n_clusters=5, dbscan_eps=1.5)
display(scores.round(3))


Silhouette is higher when groups are better separated; Davies-Bouldin is lower. DBSCAN labels `-1` as noise. Its scores exclude noise, so compare its noise fraction as well. Scores can be missing when DBSCAN finds fewer than two clusters.


## 5. Visualize and interpret the groups

PCA is used only to draw the two-dimensional chart. Models and scores above use all six scaled features.


In [ ]:
projection = PCA(n_components=2).fit_transform(scaled)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (name, labels) in zip(axes, labels_by_model.items()):
    ax.scatter(projection[:, 0], projection[:, 1], c=labels, s=5, cmap="tab10", alpha=0.6)
    ax.set_title(name)
    ax.set_xlabel("PC 1")
    ax.set_ylabel("PC 2")
plt.tight_layout()
plt.show()


In [ ]:
profile = sample[CLUSTER_FEATURES].copy()
profile["cluster"] = labels_by_model["K-Means"]
display(profile.groupby("cluster").agg(["mean", "median"]).round(2))
display(profile["cluster"].value_counts().sort_index().rename("observations"))


## Conclusion

Use the profile table to name patterns only after checking their original-unit values. These clusters are descriptive groups within one household. Try several K values and DBSCAN radii, and compare stability across random seeds, before treating any cluster count as a firm finding.
